In [ ]:
import pandas as pd
import sqlalchemy as sa
import plotly.express as px

engine = sa.create_engine("mysql+pymysql://root:@localhost/climate_watch")

# 1. Haal de data op (we gebruiken kleine letters zoals eerder vastgesteld)
query = "SELECT city, latitude, longitude, average_temp FROM cleaned_global_temp_major_city"

try:
    df_city_raw = pd.read_sql(query, engine)
    df_city_raw = df_city_raw.rename(columns={'average_temp': 'temp'})

    # 2. Verbeterde functie voor coördinaten
    def convert_coords(coord):
        if coord is None or str(coord).strip() == "": 
            return 0.0
        
        coord = str(coord).upper().strip() # Veiligheid: hoofdletters en spaties weg
        
        # Als het S of W is, maak het getal negatief
        if 'S' in coord or 'W' in coord:
            return -float(coord[:-1])
        # Als het N of E is, alleen de letter weg
        elif 'N' in coord or 'E' in coord:
            return float(coord[:-1])
        
        # Voor het geval er al pure getallen in staan
        return float(coord)

    # Toepassen op beide kolommen
    df_city_raw['lat'] = df_city_raw['latitude'].apply(convert_coords)
    df_city_raw['lon'] = df_city_raw['longitude'].apply(convert_coords)

    # 3. Groeperen en Plotten
    df_city_map = df_city_raw.groupby(['city', 'lat', 'lon'])['temp'].mean().reset_index()

    fig = px.scatter_geo(df_city_map, 
                         lat='lat', 
                         lon='lon', 
                         hover_name='city', 
                         color='temp',
                         color_continuous_scale=px.colors.sequential.YlOrRd,
                         title='Geospatiale Analyse: Gemiddelde Temperatuur per Wereldstad',
                         projection="natural earth")
    fig.show()
    
    print("De kaart is succesvol gegenereerd!")

except Exception as e:
    print(f" Er ging iets mis: {e}")